# Few-shot Example Selection

### source : PROMPT_AND_INSTRUCTION_DESIGN_2\few_shot_selection_3.ipynb**



### Objective:
Given a set of candidate few-shot examples for a Databricks agent,
evaluate each against canonical coverage criteria and select the minimal set that
improves agent performance without inflating token cost.*

## Setup

In [ ]:
import os, json, uuid, re
import sys
# Add parent directory to path to access shared utilities
sys.path.append(os.path.abspath(".."))
from _utils.sql_utils import sql_call   # shared SQL helper
from dotenv import load_dotenv
load_dotenv()   # reads DATABRICKS_HOST, TOKEN, SQL_WAREHOUSE_ID, MLFLOW_TRACKING_URI
assert os.getenv("DATABRICKS_HOST"),     "DATABRICKS_HOST not set"
assert os.getenv("DATABRICKS_TOKEN"),    "DATABRICKS_TOKEN not set"
assert os.getenv("SQL_WAREHOUSE_ID"),    "SQL_WAREHOUSE_ID not set"
assert os.getenv("MLFLOW_TRACKING_URI"), "MLFLOW_TRACKING_URI not set"

## Imports and config

In [ ]:
import mlflow, tiktoken
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks
from langchain_core.messages import HumanMessage, SystemMessage

# Unity Catalog coordinates -- prompts registered as CATALOG.SCHEMA.name
CATALOG, SCHEMA = "demo", "context"
SUFFIX          = uuid.uuid4().hex[:6]   # unique suffix per run
MODEL_ENDPOINT  = "databricks-meta-llama-3-3-70b-instruct"

# Workspace client -- credentials loaded from .env
w = WorkspaceClient()

# MLflow experiment -- links to UC prompt registry via tag
mlflow.set_experiment("/Users/oliver@mlops-media.com/chapter_2_few_shot_selection")
mlflow.set_experiment_tags({"mlflow.promptRegistryLocation": f"{CATALOG}.{SCHEMA}"})
print(f"host={w.config.host}  endpoint={MODEL_ENDPOINT}  suffix={SUFFIX}")

In [ ]:
# Invoke the serving endpoint via LangChain ChatDatabricks wrapper
llm = ChatDatabricks(endpoint=MODEL_ENDPOINT)
def call_llm(system, user, max_tokens=512):
    msgs = ([SystemMessage(content=system)] if system else []) + [HumanMessage(content=user)]
    return llm.invoke(msgs, max_tokens=max_tokens).content

# tiktoken (cl100k_base) as a cost-proxy token counter for prompt budgeting
enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text):
    return len(enc.encode(text or ""))

## 1 -- Candidate pool

In [ ]:
# Each candidate is defined with id, tags (clause + domain + complexity),
# question and reference SQL. ex3 is intentionally excluded -- it is
# redundant with ex1 (same domain: sales, same clause: GROUP BY).
CANDIDATES = [
    {
        "id": "ex1", "tags": ["GROUP BY", "sales", "medium"],
        "question": "What are the top 5 products by revenue last month?",
        "sql": (
            "SELECT product_name, SUM(revenue) AS total_revenue\n"
            "FROM sales\n"
            "WHERE order_date >= DATE_TRUNC('month', CURRENT_DATE - INTERVAL 1 MONTH)\n"
            "  AND order_date <  DATE_TRUNC('month', CURRENT_DATE)\n"
            "GROUP BY product_name ORDER BY total_revenue DESC LIMIT 5;"
        ),
    }
]

In [ ]:
CANDIDATES += [    
    {
        "id": "ex2", "tags": ["filter", "sales", "simple"],
        "question": "How many orders were placed today?",
        "sql": "SELECT COUNT(*) AS orders_today FROM orders WHERE order_date = CURRENT_DATE;",
    },
    {
        "id": "ex4", "tags": ["GROUP BY", "HAVING", "customers", "medium"],
        "question": "Which customers have a return rate above 10%?",
        "sql": (
            "SELECT customer_id,\n"
            "       COUNT(CASE WHEN status='returned' THEN 1 END)*1.0/COUNT(*) AS return_rate\n"
            "FROM orders GROUP BY customer_id\n"
            "HAVING return_rate > 0.10 ORDER BY return_rate DESC;"
        ),
    },
]

In [ ]:
CANDIDATES += [        
    {
        "id": "ex5", "tags": ["WINDOW", "JOIN", "CTE", "sales", "complex"],
        "question": "Week-over-week revenue change per product category?",
        "sql": (
            "WITH weekly AS (\n"
            "    SELECT category, DATE_TRUNC('week', order_date) AS week, SUM(revenue) AS rev\n"
            "    FROM sales JOIN products USING (product_id) GROUP BY 1, 2)\n"
            "SELECT category, week, rev,\n"
            "       LAG(rev) OVER (PARTITION BY category ORDER BY week) AS prev,\n"
            "       rev - LAG(rev) OVER (PARTITION BY category ORDER BY week) AS delta\n"
            "FROM weekly ORDER BY category, week;"
        ),
    },
    {
        "id": "ex6", "tags": ["WINDOW", "subquery", "inventory", "complex"],
        "question": "Which warehouse has the lowest stock level for each product?",
        "sql": (
            "SELECT product_id, warehouse_id, stock_level FROM (\n"
            "    SELECT *, RANK() OVER (PARTITION BY product_id ORDER BY stock_level) AS rnk\n"
            "    FROM inventory) t WHERE rnk = 1;"
        ),
    },
]
print(f"Pool: {len(CANDIDATES)} candidates")
for ex in CANDIDATES:
    print(f"  {ex['id']} {count_tokens(ex['question']+ex['sql']):>5} tok  {ex['tags']}")

## 2 -- Discriminating eval questions

Standard keyword scoring fails because table names in `SCHEMA_DEF` let the model
produce `GROUP BY`, `HAVING`, `COUNT` even with 0 examples -- all subsets score the same.


In [ ]:
# Schema made available to the agent at inference time
SCHEMA_DEF = (
    "Schema:\n"
    "  sales(order_date DATE, product_id STRING, revenue DOUBLE, customer_id STRING)\n"
    "  orders(order_id STRING, order_date DATE, customer_id STRING, status STRING,"
    " discount_amount DOUBLE)\n"
    "  products(product_id STRING, product_name STRING, category STRING)\n"
    "  inventory(product_id STRING, warehouse_id STRING, stock_level INT)"
)


In [ ]:
# Each question lists required patterns, forbidden workarounds, and tables
EVAL_QS = [
    {   # Q1 -- LAG() OVER: only taught by ex5
        "q": "For each product category, show this month's revenue and last month's side by side.",
        "must": ["LAG(", "OVER (PARTITION BY", "DATE_TRUNC"],
        "bad": [r"JOIN\s+\w+.*ON.*JOIN"],
        "tables": ["sales", "products"], "wp": 0.5, "wt": 0.5,
    },
    {   # Q2 -- HAVING + ratio: only taught by ex4
        "q": "Which customers placed >5 orders AND have cancellation rate >20%? Show rate.",
        "must": ["HAVING", "COUNT(", "/ COUNT("],
        "bad": [r"WHERE.*cancel"],
        "tables": ["orders"], "wp": 0.6, "wt": 0.4,
    },
    {   # Q3 -- RANK() OVER in subquery: only taught by ex6
        "q": "Per product: warehouse_id and stock of the LOWEST-stock warehouse. Include ties.",
        "must": ["RANK()", "OVER (PARTITION BY", "WHERE"],
        "bad": [r"MIN\(stock"],
        "tables": ["inventory"], "wp": 0.6, "wt": 0.4,    
    },
]

In [ ]:
# Structural scorer: checks required patterns, penalises naive workarounds
# Scores the GENERATED SQL only -- not the system prompt -- to avoid false positives
def score_sql(sql, q):
    u = sql.upper().replace("(", " ").replace(")", " ")
    ps = sum(p.upper().replace("(","").replace(")","").strip() in u
             for p in q["must"]) / len(q["must"])
    if any(re.search(b, sql, re.I) for b in q["bad"]): ps = min(ps, 0.3)
    ts = sum(re.search(rf"\b{t}\b", sql, re.I) is not None
             for t in q["tables"]) / len(q["tables"])
    return round(q["wp"]*ps + q["wt"]*ts, 2)

# Few-shot prompt builder: injects selected examples before the user question
def build_prompt(examples):
    shots = "".join(f"Q: {e['question']}\nSQL:\n{e['sql']}\n\n" for e in examples)
    return (f"SQL assistant for a retail data warehouse.\n{SCHEMA_DEF}\n\n"
            "Return ONLY SQL.\n\n"
            f"{('Examples:\n'+shots) if shots else 'No examples.'}")

print("Helpers ready")
for i, q in enumerate(EVAL_QS):
    print(f"  Q{i+1}: {q['q'][:70]}...")

## 3 -- Manual grid: 0-shot to full pool

**Redundant 3-shot (ex1+ex2+ex4):** teaches GROUP BY, filter, HAVING -- none teaches
`LAG()` or `RANK()`. Token cost rises but Q1 and Q3 scores stay at 0-shot level.

**Optimal 3-shot (ex4+ex5+ex6):** each example teaches exactly one required construct.
Minimum tokens, maximum eval coverage.

Expected: `0-shot` ≈ `redundant` on Q1/Q3; `optimal` scores significantly higher.

In [ ]:
# eval_subset: builds prompt, calls LLM on each eval question, logs to MLflow
# Returns per-question scores to reveal which examples help which constructs
def eval_subset(ids, run_name):
    subset = [e for e in CANDIDATES if e["id"] in ids]
    prompt = build_prompt(subset)
    ptoks  = count_tokens(prompt)
    per_q  = [score_sql(call_llm(prompt, q["q"], max_tokens=350), q) for q in EVAL_QS]
    avg    = round(sum(per_q)/len(per_q), 3)
    s1k    = round(avg/ptoks*1000, 4)
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("ids", ",".join(ids) or "none")
        mlflow.log_metric("tokens", ptoks)
        mlflow.log_metric("avg", avg)
        mlflow.log_metric("s1k", s1k)
        [mlflow.log_metric(f"q{i+1}", s) for i, s in enumerate(per_q)]
    return {"ids": ids, "tokens": ptoks, "avg": avg, "s1k": s1k, "per_q": per_q}



In [ ]:
# 4 subsets designed to show the coverage matrix effect
SUBSETS = {
    "0-shot":                          [],
    "redundant 3-shot (ex1+ex2+ex4)":  ["ex1","ex2","ex4"],
    "optimal 3-shot  (ex4+ex5+ex6)":   ["ex4","ex5","ex6"],
    "full 5-shot":                     ["ex1","ex2","ex4","ex5","ex6"],
}
results_B = {}
for name, ids in SUBSETS.items():
    print(f"  {name} ...")
    results_B[name] = eval_subset(ids, f"B_{name[:12]}_{SUFFIX}")

In [ ]:
print(f"\n{'Subset':<35} {'Tok':>6} {'Avg':>6} {'S/1k':>7}  Q1    Q2    Q3")
print("-"*72)
for n, r in results_B.items():
    qs = "  ".join(f"{s:.2f}" for s in r["per_q"])
    print(f"{n:<35} {r['tokens']:>6} {r['avg']:>6.3f} {r['s1k']:>7.4f}  {qs}")

## 4 -- DSPy-style greedy selection

Mirrors **BootstrapFewShot / MIPROv2**: at each step the LLM scores remaining
candidates against 4 coverage criteria + a redundancy penalty, then picks the best.
Stops early when marginal gain drops below 0.05.

In [ ]:
# Coverage criteria mirror the exam canonical criteria
CRITERIA = [
    "SQL clause diversity (GROUP BY, WINDOW, JOIN, subquery, filter)",
    "Business domain coverage (sales, inventory, customers, returns)",
    "Complexity gradient (simple / medium / complex)",
    "Non-redundancy with already-selected examples",
]

In [ ]:

def score_candidate(c, selected):
    # Summarise already-selected to give the LLM context for redundancy check
    sel = json.dumps([{"id":e["id"],"tags":e["tags"]} for e in selected]) if selected else "none"
    p = ("Evaluate a few-shot SQL example for an agent prompt.\n"
         "Criteria:\n" + "\n".join(f"- {x}" for x in CRITERIA)
         + f"\nAlready selected: {sel}\n"
         f"Candidate id={c['id']} tags={c['tags']}\nQ: {c['question']}\nSQL: {c['sql']}\n\n"
         "Return ONLY JSON: {coverage_score, redundancy_penalty, net_score, reason}")
    raw = call_llm("", p, 200).replace("```json","").replace("```","").strip()
    r = json.loads(raw); r["id"] = c["id"]; return r

# Greedy forward pass: pick highest net_score at each step
def greedy_select(pool, max_k=3):
    selected, remaining = [], pool.copy()
    for step in range(max_k):
        print(f"\nStep {step+1}:")
        scored = [score_candidate(c, selected) for c in remaining]
        [print(f"  {s['id']}: net={s['net_score']:.2f} -- {s['reason']}") for s in scored]
        best = max(scored, key=lambda s: s["net_score"])
        if best["net_score"] <= 0.05: print("  Early stop"); break
        chosen = next(c for c in remaining if c["id"]==best["id"])
        selected.append(chosen)
        remaining = [c for c in remaining if c["id"]!=best["id"]]
        print(f"  Selected: {best['id']}")
    return selected

In [ ]:
auto = greedy_select(CANDIDATES, max_k=3)
auto_ids = [e["id"] for e in auto]
print(f"\nAuto-selected: {auto_ids}")
auto_r = eval_subset(auto_ids, f"B_auto_{SUFFIX}")
results_B["auto 3-shot"] = auto_r
print(f"avg={auto_r['avg']:.3f}  tokens={auto_r['tokens']}  s/1k={auto_r['s1k']:.4f}")